# PPIFlow Protein Design

<img src="https://github.com/Mingchenchen/PPIFlow/blob/main/model.png?raw=true">

# Demo outline

1. PPIFlow backbone generation
2. MPNN Squence Design
3. Flowpacker Sidechain packing

In [0]:
%pip install torch-scatter==2.1.2

In [0]:
%restart_python

In [0]:
#@title 1. PPIFlow backbone generation

#@markdown --- **Configuration Parameters** ---

#@markdown **Antigen PDB Path:** Path to the antigen PDB file in Google Drive. This file contains the protein the nanobody will bind to.
antigen_pdb = "../targets/1CVS.pdb" #@param {type:"string"}
#@markdown **Framework PDB Path:** Path to the framework PDB file in Google Drive. This file provides the structural scaffold for the nanobody.
framework_pdb = "../framework/7eow_nanobody_framework.pdb" #@param {type:"string"}
#@markdown **Antigen Chain ID:** The specific chain identifier within the antigen PDB file that the nanobody targets.
antigen_chain = "C" #@param {type:"string"}
#@markdown **Heavy Chain ID:** The chain identifier for the heavy chain of the nanobody within the Framework PDB.
heavy_chain = "A" #@param {type:"string"}
#@markdown **Specified Hotspots:** A comma-separated list of residues on the antigen that are crucial for binding. These are usually in the format 'ChainIDResidueNumber'.
specified_hotspots = "C101,C135,C171,C198" #@param {type:"string"}
#@markdown **CDR Lengths:** Specifies the desired lengths for the Complementarity Determining Regions (CDRs). Format: 'CDRName,MinLength-MaxLength'.
cdr_length = "CDRH1,8-8,CDRH2,8-8,CDRH3,9-21" #@param {type:"string"}
#@markdown **Samples Per Target:** The number of nanobody backbone samples to generate for each target.
samples_per_target = 3 #@param {type:"integer"}
#@markdown **Config Path:** Path to the YAML configuration file for the PPIFlow model.
config_path = "../configs/inference_nanobody.yaml" #@param {type:"string"}
#@markdown **Model Weights Path:** Path to the pre-trained model weights (.ckpt file) for the nanobody generation model.
model_weights = "/Volumes/sandbox/model_weights/protein_hunter/nanobody.ckpt" #@param {type:"string"}
#@markdown **Output Directory:** Directory where the generated PDB files will be saved.
output_dir = "/tmp/nanobody_test2" #@param {type:"string"}
#@markdown **Output Name Prefix:** A prefix to use for the names of the generated output files.
name = "1CVS_FGFR1" #@param {type:"string"}

# Run via subprocess
import subprocess
subprocess.run(["python", "../sample_antibody_nanobody.py", 
                "--antigen_pdb", antigen_pdb, 
                "--framework_pdb", framework_pdb, 
                "--antigen_chain", antigen_chain, 
                "--heavy_chain", heavy_chain, 
                "--specified_hotspots", specified_hotspots, 
                "--cdr_length", cdr_length, 
                "--samples_per_target", str(samples_per_target), 
                "--config", config_path, 
                "--model_weights", model_weights, 
                "--output_dir", output_dir, 
                "--name", name])


In [0]:
import shutil
# 1. Define paths
job_name = output_dir.split("/")[-1]
volume_save_path = f'/Volumes/sandbox/denovotrial/ppiflow/{job_name}'

# 2. Use shutil.copytree instead of dbutils
# dirs_exist_ok=True allows it to overwrite/merge if the folder already exists
shutil.copytree(output_dir, volume_save_path, dirs_exist_ok=True)

In [0]:
#@title Display backbone
import py3Dmol

# 1. 读取 PDB 文件
pdb_path = f"{volume_save_path}/1CVS_FGFR1_0.pdb" #@param {type:"string"}

with open(pdb_path, 'r') as f:
    pdb_content = f.read()

# 2. 创建查看器
# 设置背景为白色以增加对比度，方便观察 B-factor 颜色
view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_content, 'pdb')

# 3. 为 A 链和 C 链设置基于 B-factor 的着色样式
# 'prop': 'b' 指代 B-factor
# 'gradient': 'roygb' (红-橙-黄-绿-蓝) 是经典的冷热图映射
# 通常高 B-factor（不稳定/柔性）显示为红色，低 B-factor（稳定）显示为蓝色
style_config = {
    'cartoon': {
        'colorscheme': {
            'prop': 'b',
            'gradient': 'roygb',
            'min': 0,    # 可选：手动设置 B-factor 范围以统一色阶
            'max': 5   # 可选：根据实际数据调整
        }
    }
}

# 分别应用到两条链
view.setStyle({'chain': 'A'}, style_config)
view.setStyle({'chain': 'C'}, style_config)

# 4. 辅助视觉效果
view.setBackgroundColor('white') # 设置白色背景
view.zoomTo()                    # 自动缩放以适应模型

# 5. 显示
view.show()

In [0]:
#@title 2. MPNN Squence Design
import subprocess

abmpnn_checkpoint ="/Volumes/sandbox/model_weights/protein_hunter"
model_name = "abmpnn"
folder_with_pdbs =f"{volume_save_path}"
mpnn_dir =f"{volume_save_path}/nanobody_mpnn"
position_fixed =f"{volume_save_path}/fixed_positions.csv"
chains_to_design ="A"
num_seq_per_target=8
sampling_temp =0.5
seed = 37

# Run setup script
subprocess.run(['python', '../demo_scripts/make_fix_csv.py', volume_save_path])

# Run actual protein_mpnn_run script
subprocess.run(["python", "../ProteinMPNN/protein_mpnn_run.py", 
                "--path_to_model_weights", abmpnn_checkpoint, 
                "--model_name", model_name, 
                "--folder_with_pdbs", folder_with_pdbs, 
                "--out_folder", mpnn_dir, 
                "--chain_list", chains_to_design, 
                "--position_list", position_fixed, 
                "--num_seq_per_target", str(num_seq_per_target), 
                "--sampling_temp", str(sampling_temp), 
                "--seed", str(seed), 
                "--batch_size", str(num_seq_per_target), 
                "--omit_AAs", "C"])

In [0]:
seq_file = f"{mpnn_dir}/seqs/1CVS_FGFR1_0.fa"
!cat {seq_file} | head

In [0]:
#@title 3. Flowpacker Sidechain packing

import os
import csv
from Bio import SeqIO

def direct_fasta_to_csv(input_dirs: list, output_csv: str, suffix: str = ".pdb"):

    seen_seqs = set()

    os.makedirs(os.path.dirname(os.path.abspath(output_csv)), exist_ok=True)

    with open(output_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["link_name", "seq", "seq_idx"])
        for folder in input_dirs:
            if not os.path.exists(folder): continue
            files = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(('.fasta', '.fa'))]

            for file_path in files:
                base_name = os.path.splitext(os.path.basename(file_path))[0]

                for i, record in enumerate(SeqIO.parse(file_path, "fasta")):
                    if i == 0:
                        continue

                    seq_str = str(record.seq)
                    if seq_str in seen_seqs:
                        continue
                    seen_seqs.add(seq_str)
                    link_name = f"{base_name}{suffix}"
                    seq_idx = str(i)

                    writer.writerow([link_name, seq_str, seq_idx])

    print(f"✅ Processing complete! {len(seen_seqs)} unique sequences have been written to: {output_csv}")



fa_folders = [
    '/content/nanobody_test_mpnn/seqs',
]

direct_fasta_to_csv(
    input_dirs=fa_folders,
    output_csv=f'/content/nanobody_test_mpnn/final_result.csv',
    suffix=".pdb"
)

In [0]:
#@title 3.1 Install flowpacker
%%bash
export PATH="/usr/local/miniconda3/condabin:$PATH"

# Accept Conda Terms of Service
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Initialize conda for the current shell session to make 'conda activate' available
source /usr/local/miniconda3/etc/profile.d/conda.sh

git clone https://gitlab.com/mjslee0921/flowpacker.git
mv /content/flowpacker /content/flowpacker-main

conda env create -f /content/PPIFlow/demo_scripts/flowpacker_af3score/environment.yml
conda activate flowpacker

In [0]:
#@title 3.2 Run flowpacker
%%bash
export PATH="/usr/local/miniconda3/condabin:$PATH"

# Accept Conda Terms of Service
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Initialize conda for the current shell session to make 'conda activate' available
source /usr/local/miniconda3/etc/profile.d/conda.sh

conda activate flowpacker

cp /content/PPIFlow/demo_scripts/flowpacker_af3score/sampler_pdb_colab.py /content/flowpacker-main/

CSV_FILE=/content/nanobody_test_mpnn/final_result.csv
FP_BASE=/content/flowpacker_out
FP_LOGS="$FP_BASE/logs"
FP_OUTS="$FP_BASE/outputs"

mkdir -p "$FP_OUTS" "$FP_LOGS"
echo "Running Flowpacker..."
cd /content/flowpacker-main

python sampler_pdb_colab.py base \
--pdb_dir /content/nanobody_test \
--save_dir "$FP_OUTS" \
--use_gt_masks True \
--csv_file "$CSV_FILE"

echo "------------------------------"
echo "Flowpacker out PDB:"
echo "${FP_BASE}/after_pdbs"

In [0]:
#@title 3.3 Display Design

import py3Dmol

# 1. 读取 PDB 文件
pdb_path = "/content/flowpacker_out/after_pdbs/1CVS_FGFR1_0_1.pdb" #@param {type:"string"}

with open(pdb_path, 'r') as f:
    pdb_content = f.read()

# 2. 创建查看器
# 设置背景为白色以增加对比度，方便观察 B-factor 颜色
view = py3Dmol.view(width=800, height=600)
view.addModel(pdb_content, 'pdb')

# 分别应用到两条链
view.setStyle({'chain': 'A'}, style_config)
view.setStyle({'chain': 'C'}, style_config)

# 4. 辅助视觉效果
view.setBackgroundColor('white') # 设置白色背景
view.zoomTo()                    # 自动缩放以适应模型

# 5. 显示
view.show()